# ThinCurr Python Example: One Filament Reconstruction Using LSTSQ

In this notebook we demonstrate how to use the jamfit.py script for filament reconstruction, this is the follow up to the one_filament_create_ex

## Load ThinCurr Library

To load the ThinCurr filaments python script, you will need to define where ThinCurr is located. This can be done through the PYTHONPATH enviornment variable or within the script using sys.path.append() as shown below, where we look for the environment variable OFT_ROOTPATH to procide the path to where the OpenFUSIONToolkit is installed. If this enviornment variable is not defined, you can do so within the script as well by setting:
 ```os.environ["OFT_ROOTPATH"] = \path\to\OpenFUSIONToolkit\install_release```

In [ ]:
import os 
import sys
import numpy as np
import matplotlib.pyplot as plt
thincurr_python_path = os.getenv('OFT_ROOTPATH')
if thincurr_python_path is not None:
    sys.path.append(os.path.join(thincurr_python_path,'python'))
from OpenFUSIONToolkit._core import OFT_env
from OpenFUSIONToolkit.ThinCurr import ThinCurr
from OpenFUSIONToolkit.ThinCurr.jamfit import Jamfit
from OpenFUSIONToolkit.ThinCurr.sensor import Mirnov, save_sensors, circular_flux_loop

## Intialize Jamfit Object and Load in Data

You will need some reduced object to plug into Jamfit. You can use the intialize reduced object function within Jamfit to do so. In this case we will use the reduced object we created within the last notebook. 

In [ ]:
myOFT = OFT_env(nthreads=4)
meshfile_thincurr = "thincurr_ex-ports.h5"
xml_name = "oft_in.xml"
reduced_filename = "reduced_model.h5"
jam_obj = Jamfit(xml_name, meshfile_thincurr, oft_env=myOFT)
jam_obj.initialize_reduced_model(reduced_filename)

magnetic_data = np.load('sensor_measurements.npy')
time_array_sensors = np.load('time_array_sensors.npy')
time_array_coils = np.load('time_array_coilcurr.npy')
totalip = np.load('totalip.npy')
coilcurrs = np.load('coil_currs.npy')

num_coils = 1
wall_modes = 3

## Running Jamfit's LSTSQ Reconstruction Algorithm

Here we run Jamfit's built-in least-squares reconstruction algorithm (more detailed comments on the algorithm can be found within the class).

> **Note:** To run a full shot, call `run_reconstruction_lstsq` in a loop over time.

### Parameters

| Parameter | Type | Description |
|---|---|---|
| `Psi_at_time` | `np.ndarray` | Sensor flux measurements at time |
| `ip_at_time` | `float` | Total plasma current measurement at time |
| `num_non_fil_coils` | `int` | Number of non-filament coils in the system |
| `ip_weight` | `float` | Weight for the total plasma current in the reconstruction |
| `sigma` | `np.ndarray` | Standard deviations for each sensor measurement (used for weighting) |
| `reg_factor_fil` | `float` | Regularization factor for filament currents |
| `reg_factor_wall` | `float` | Regularization factor for wall currents |

In [ ]:
rms_global = np.sqrt(np.mean(magnetic_data**2))
sigma = np.maximum(0.05 * np.abs(magnetic_data).mean(axis=0), 0.02 * rms_global)

solution_tidx = [] 
residual_tidx = [] 

ip_weight = 1E3

for t_idx in range(len(time_array_sensors)):
    psi_at_time = magnetic_data[t_idx, :]
    ip_at_time = totalip[t_idx]
    coil_curr_at_time = coilcurrs[t_idx, :]

    solution, residual, Ax, B = jam_obj.run_reconstruction_lstsq(psi_at_time, ip_at_time, num_coils, coil_curr_at_time, ip_weight, sigma, reg_factor_fil=1E-100 ,reg_factor_wall=1E-100)
    solution_tidx.append(solution)
    residual_tidx.append(residual)


### Sweeping Over the IP Regularization Term

Picking regularization terms is often the hardest part of using `run_reconstruction_lstsq`. 
This cell sweeps over a range of `ip_weight` values and plots the resulting residuals. 
As expected, increasing `ip_weight` increases the residual but decreases the IP error.

### Choosing `reg_factor_fil` and `reg_factor_wall`

For the filament and wall regularization factors, users can perform an **L-curve sweep**, which is a log-log 
plot of `||Ax - b||` versus `||x||`. The ideal regularization term sits at the corner 
of the L-curve. 

In this problem we do not need to do L-curve regularization as regularization is not necessary for this case (this is why they are set at extremely low values), however for more complex problems it may be necessary to do so. 

> **Note:** Problems that are not discrete ill-posed may not have a distinctive L-curve corner.


In [ ]:
ip_reg_list = np.logspace(-6, 10, 12)
avg_residual_list = []
ip_error_list = [] 
for ip_reg in ip_reg_list:
    solution_tidx = [] 
    residual_tidx = [] 
    for t_idx in range(len(time_array_sensors)):
        psi_at_time = magnetic_data[t_idx, :]
        ip_at_time = totalip[t_idx]
        coil_curr_at_time = coilcurrs[t_idx, :]

        solution_fil, solution_wall,  residual, Ax, B = jam_obj.run_reconstruction_lstsq(psi_at_time, ip_at_time, num_coils, coil_curr_at_time, ip_reg, sigma, reg_factor_fil=1E-100, reg_factor_wall=1E-100)
        solution = np.concatenate([solution_wall, solution_fil])
        solution_tidx.append(solution)
        residual_tidx.append(residual)
    avg_residual = np.mean(residual_tidx)
    avg_residual_list.append(avg_residual)
    ip_error = np.mean(np.abs([solution[-1] - ip_at_time for solution, ip_at_time in zip(solution_tidx, totalip)]))
    ip_error_list.append(ip_error)
plt.loglog(ip_reg_list, avg_residual_list)
plt.xlabel('IP Regularization Factor')
plt.ylabel('Average Residual')
plt.title('Average Residual vs IP Regularization Factor')
plt.grid()
plt.show()
plt.loglog(ip_reg_list, ip_error_list)
plt.xlabel('IP Regularization Factor')
plt.ylabel('Average IP Error')
plt.title('Average IP Error vs IP Regularization Factor')
plt.grid() 
plt.show()


# Plotting Results

Here we plot the results. 

We can see that the plasma filament matches the expected plasma current.  

In [ ]:
plasma_fil = np.array([sol[-1] for sol in solution_tidx])
plt.plot(time_array_sensors, residual_tidx, marker='o')
plt.xlabel('Time Step')
plt.ylabel('Residual')
plt.title('Reconstruction Residual Over Time')
plt.grid()
plt.show() 
plt.figure() 
plt.plot(time_array_sensors, plasma_fil, 'r')
plt.plot(time_array_sensors, totalip, '--', label='True Total IP', color='k')
plt.xlabel('Time Step')
plt.ylabel('Reconstructed Filament Current')
plt.title('Reconstructed Filament Current Over Time')
plt.grid()
plt.show()

### Comparing wall modes
In the following cell you can see we plot the wall modes and see that it matches the expected wall mode weights that we saw in the previous notebook. 

In [ ]:
num_Ms = jam_obj.torus_reduced.Ms.shape[0]
temp_curr = np.array(solution_tidx)[:, 0:num_Ms]  # shape (time, num_modes)
max_weights = [abs(temp_curr[:, i]).sum() for i in range(temp_curr.shape[1])] # total sum over time 


top_modenum_indices = sorted(range(len(max_weights)), key=lambda i: max_weights[i], reverse=True)[:wall_modes]
for i in top_modenum_indices:
    print(f"Mode {i}: Total weight = {max_weights[i]:.4f}")


for i in range(temp_curr.shape[1]):
    plt.plot(range(temp_curr.shape[0]), temp_curr[:, i], marker='o', label=f'Mode {i}')
plt.xlabel('Time Step')
plt.ylabel('Mode Amplitude')
plt.legend(loc = 'lower left')
plt.title('Top Mode Amplitudes Over Time')
plt.show()